In [1]:
import pandas as pd
import numpy as np
from IPython.display import display 

print("Veri setleri yükleniyor.")

try:
    metadata_df = pd.read_csv('movies_metadata.csv', low_memory=False)
    print("movies_metadata.csv başarıyla yüklendi.")
except FileNotFoundError:
    print("movies_metadata.csv bulunamadı.")
    
try:
    ratings_df = pd.read_csv('ratings.csv')
    print("ratings.csv başarıyla yüklendi.")
except FileNotFoundError:
    print("ratings.csv bulunamadı.")
    
if 'metadata_df' in locals():
    print(f"\nMeta Veri Satır Sayısı: {metadata_df.shape[0]}")
    display(metadata_df[['id', 'title', 'overview', 'genres']].head()) 
    
if 'ratings_df' in locals():
    print(f"\nRatings Veri Satır Sayısı: {ratings_df.shape[0]}")
    display(ratings_df[['userId','movieId','rating','timestamp']].head())

Veri setleri yükleniyor.
movies_metadata.csv başarıyla yüklendi.
ratings.csv başarıyla yüklendi.

Meta Veri Satır Sayısı: 45466


,id,title,overview,genres
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]"



Ratings Veri Satır Sayısı: 26024289


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

print("ID dönüştürme işlemi ve veri birleştirme işlemi yapılıyor.")

metadata_df['id'] = pd.to_numeric(metadata_df['id'], errors='coerce') 
metadata_df = metadata_df.dropna(subset=['id'])
metadata_df['id'] = metadata_df['id'].astype(int)

ratings_df['movieId'] = ratings_df['movieId'].astype(int)

merged_df = pd.merge(ratings_df, 
                     metadata_df[['id', 'title', 'overview', 'genres']], 
                     left_on='movieId',     
                     right_on='id',      
                     how='inner') 

merged_df = merged_df.drop(columns=['id'])

print(f"Birleştirme Tamamlandı.")
print(f"Birleştirilmiş Veri Boyutu: {merged_df.shape}")

print("\nBirleştirilmiş Veri Tablosu")
display(merged_df[['userId', 'title', 'rating', 'overview', 'genres']].head())

ID dönüştürme işlemi ve veri birleştirme işlemi yapılıyor.
Birleştirme Tamamlandı.
Birleştirilmiş Veri Boyutu: (11437637, 7)

Birleştirilmiş Veri Tablosu


,userId,title,rating,overview,genres
0,1,Three Colors: Red,1.0,Red This is the third film from the trilogy by...,"[{'id': 18, 'name': 'Drama'}, {'id': 9648, 'na..."
1,1,The 400 Blows,4.5,"For young Parisian boy Antoine Doinel, life is...","[{'id': 18, 'name': 'Drama'}]"
2,1,Sleepless in Seattle,5.0,A young boy who tries to set his dad up on a d...,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
3,1,Rocky Balboa,5.0,When he loses a highly publicized virtual boxi...,"[{'id': 18, 'name': 'Drama'}]"
4,1,Fools Rush In,4.0,Alex Whitman (Matthew Perry) is a designer fro...,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name..."


In [3]:
print("Gereksiz sütunlar temizleniyor")

merged_df = merged_df.drop(columns=['timestamp'])

merged_df = merged_df.drop(columns=['movieId'])

print("Temizlik başarılı.")
display(merged_df.head())

Gereksiz sütunlar temizleniyor
Temizlik başarılı.


,userId,rating,title,overview,genres
0,1,1.0,Three Colors: Red,Red This is the third film from the trilogy by...,"[{'id': 18, 'name': 'Drama'}, {'id': 9648, 'na..."
1,1,4.5,The 400 Blows,"For young Parisian boy Antoine Doinel, life is...","[{'id': 18, 'name': 'Drama'}]"
2,1,5.0,Sleepless in Seattle,A young boy who tries to set his dad up on a d...,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
3,1,5.0,Rocky Balboa,When he loses a highly publicized virtual boxi...,"[{'id': 18, 'name': 'Drama'}]"
4,1,4.0,Fools Rush In,Alex Whitman (Matthew Perry) is a designer fro...,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name..."


In [4]:
print("\nEksik overview ve genres bilgisi olan satırlar temizleniyor.")

merged_df.dropna(subset=['overview', 'genres'], inplace=True)

merged_df = merged_df[merged_df['overview'].str.strip() != '']
merged_df = merged_df[merged_df['genres'].str.strip() != '']


print(f"Kalan satır sayısı: {merged_df.shape[0]}")


Eksik overview ve genres bilgisi olan satırlar temizleniyor.
Kalan satır sayısı: 11396944


In [5]:
from IPython.display import display

SIZE = 10000 
if merged_df.shape[0] > SIZE:
    final_df = merged_df.sample(n=SIZE, random_state=42).copy()
else:
    final_df = merged_df.copy()

print(f"Veri satır sayısı: {final_df.shape[0]}")

final_df.dropna(subset=['overview', 'genres'], inplace=True)
final_df = final_df[final_df['overview'].str.strip() != '']
final_df = final_df[final_df['genres'].str.strip() != '']

print(f"LLM Eğitim Verisi Boyutu: {final_df.shape[0]}")

display(final_df[['userId', 'title', 'overview', 'genres', 'rating']].head()) 

Veri satır sayısı: 10000
LLM Eğitim Verisi Boyutu: 10000


,userId,title,overview,genres,rating
2128305,50034,The Holy Mountain,A Mexican master leads a Christ figure and oth...,"[{'id': 18, 'name': 'Drama'}]",3.0
4236792,99717,Sweet Sixteen,Determined to have a normal family life once h...,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",4.0
2239175,52690,Rope,"Two young men strangle their ""inferior"" classm...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",4.0
9526639,225128,Within the Woods,The low budget film starring the young Bruce C...,"[{'id': 27, 'name': 'Horror'}]",4.0
11163381,264400,7 Virgins,Tano is 16-years-old and is already sitting in...,"[{'id': 18, 'name': 'Drama'}]",2.0


In [6]:
from IPython.display import display 

print("\nLLM için Prompt Response Oluşturuluyor.") #giçı
final_df['prompt'] = (
    "Kullanıcının izlediği filmin özeti: " + 
    final_df['overview'] + 
    ", Türleri: " + 
    final_df['genres'].astype(str) + 
    "."
)
final_df['response'] = final_df['title']
print("Prompt ve Response çiftleri başarıyla oluşturuldu.")
display(final_df[['prompt', 'response']].head())


LLM için Prompt Response Oluşturuluyor.
Prompt ve Response çiftleri başarıyla oluşturuldu.


,prompt,response
2128305,Kullanıcının izlediği filmin özeti: A Mexican ...,The Holy Mountain
4236792,Kullanıcının izlediği filmin özeti: Determined...,Sweet Sixteen
2239175,Kullanıcının izlediği filmin özeti: Two young ...,Rope
9526639,Kullanıcının izlediği filmin özeti: The low bu...,Within the Woods
11163381,Kullanıcının izlediği filmin özeti: Tano is 16...,7 Virgins


In [7]:
print("Prompt metninin tamamı")
print(final_df['prompt'].iloc[0])

Prompt metninin tamamı
Kullanıcının izlediği filmin özeti: A Mexican master leads a Christ figure and other disciples to a mountain of immortal wise men., Türleri: [{'id': 18, 'name': 'Drama'}].


In [8]:
print("GRUPLAMA") 

user_prompts = final_df.groupby('userId')['prompt'].apply(lambda x: ' | '.join(x)).reset_index()
user_prompts.rename(columns={'prompt': 'user_history_prompt'}, inplace=True)

user_responses = final_df.groupby('userId')['response'].apply(lambda x: x.sample(1).iloc[0]).reset_index()
user_responses.rename(columns={'response': 'target_response'}, inplace=True)

training_data_df = pd.merge(user_prompts, user_responses, on='userId')
training_data_df['final_prompt'] = training_data_df['user_history_prompt']

print("Kullanıcı Bazlı Eğitim Verisi Hazırlandı.")
print(f"Final eğitim veri setindeki benzersiz kullanıcı sayısı: {training_data_df.shape[0]}")
display(training_data_df[['userId', 'final_prompt', 'target_response']].head())

GRUPLAMA
Kullanıcı Bazlı Eğitim Verisi Hazırlandı.
Final eğitim veri setindeki benzersiz kullanıcı sayısı: 9276


,userId,final_prompt,target_response
0,18,Kullanıcının izlediği filmin özeti: After a gl...,Nausicaä of the Valley of the Wind
1,30,Kullanıcının izlediği filmin özeti: From an ex...,Monsoon Wedding
2,34,Kullanıcının izlediği filmin özeti: A wealthy ...,Jurassic Park
3,229,Kullanıcının izlediği filmin özeti: A film of ...,Yankee Doodle Dandy
4,235,Kullanıcının izlediği filmin özeti: Out-of-con...,Bad Boys II


In [9]:
from sklearn.model_selection import train_test_split
import pandas as pd

print("Train, Validation, Test")

train_val_df, test_df = train_test_split(training_data_df, test_size=0.20, random_state=42) 

train_df, val_df = train_test_split(train_val_df, test_size=(0.10/0.80), random_state=42) 
print(f"Toplam Veri Seti: {training_data_df.shape[0]} kullanıcı")
print("Bölme Sonuçları")
print(f"Eğitim Seti (train_df): {train_df.shape[0]} kullanıcı")
print(f"Doğrulama Seti (val_df): {val_df.shape[0]} kullanıcı") 
print(f"Test Seti (test_df): {test_df.shape[0]} kullanıcı")

print("\nLLM Veri Formatlama Tamamlandı.")

Train, Validation, Test
Toplam Veri Seti: 9276 kullanıcı
Bölme Sonuçları
Eğitim Seti (train_df): 6492 kullanıcı
Doğrulama Seti (val_df): 928 kullanıcı
Test Seti (test_df): 1856 kullanıcı

LLM Veri Formatlama Tamamlandı.


In [10]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("T5-Base Modeli ve Tokenizer Yükleniyor")

MODEL_NAME = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model ({MODEL_NAME}) başarıyla yüklendi.")
print(f"Kullanılacak Donanım: {device}")

T5-Base Modeli ve Tokenizer Yükleniyor


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Model (t5-base) başarıyla yüklendi.
Kullanılacak Donanım: cuda


In [15]:
from datasets import Dataset

def preprocess_function(examples):
    inputs = ["recommend movie: " + doc for doc in examples["final_prompt"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    
    with tokenizer.as_target_tokenizer(): 
        labels = tokenizer(examples["target_response"], max_length=64, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)

print("Tokenize işlemi bitti. Veriler artık sayısal formatta.")

Map:   0%|          | 0/6492 [00:00<?, ? examples/s]

C:\Users\User\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/928 [00:00<?, ? examples/s]

✅ Tokenize işlemi bitti. Veriler artık sayısal formatta!


In [21]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    TrainerCallback
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_film_onerici_final",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=3e-5,
    num_train_epochs=3,

    fp16=False,
    gradient_checkpointing=True,

    logging_strategy="steps",
    logging_steps=50,
    report_to="none"
)
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs:
            print(f" Step {state.global_step} | Train Loss: {logs['loss']:.4f}")
        if "eval_loss" in logs:
            print(f"Eval Loss: {logs['eval_loss']:.4f}")

In [22]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[PrintLossCallback()],
)

trainer.train()


C:\Users\User\AppData\Local\Temp\ipykernel_16544\3838835029.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,0.312200,0.293805
2,0.298900,0.273387
3,0.283500,0.267186


🔥 Step 50 | Train Loss: 0.9862
🔥 Step 100 | Train Loss: 0.6232
🔥 Step 150 | Train Loss: 0.5162
🔥 Step 200 | Train Loss: 0.4286
🔥 Step 250 | Train Loss: 0.3660
🔥 Step 300 | Train Loss: 0.3372
🔥 Step 350 | Train Loss: 0.3350
🔥 Step 400 | Train Loss: 0.3122
🧪 Eval Loss: 0.2938
🔥 Step 450 | Train Loss: 0.3191
🔥 Step 500 | Train Loss: 0.3128
🔥 Step 550 | Train Loss: 0.3014
🔥 Step 600 | Train Loss: 0.2887
🔥 Step 650 | Train Loss: 0.2984
🔥 Step 700 | Train Loss: 0.2889
🔥 Step 750 | Train Loss: 0.2933
🔥 Step 800 | Train Loss: 0.2989
🧪 Eval Loss: 0.2734
🔥 Step 850 | Train Loss: 0.2878
🔥 Step 900 | Train Loss: 0.2897
🔥 Step 950 | Train Loss: 0.2941
🔥 Step 1000 | Train Loss: 0.2812
🔥 Step 1050 | Train Loss: 0.2812
🔥 Step 1100 | Train Loss: 0.2937
🔥 Step 1150 | Train Loss: 0.2917
🔥 Step 1200 | Train Loss: 0.2835
🧪 Eval Loss: 0.2672


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1218, training_loss=0.3575797358952915, metrics={'train_runtime': 5064.8874, 'train_samples_per_second': 3.845, 'train_steps_per_second': 0.24, 'total_flos': 2635916925468672.0, 'train_loss': 0.3575797358952915, 'epoch': 3.0})

In [25]:
def film_tahmin_et(ozet):
    model.eval()
    input_text = "recommend movie: " + ozet
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"], 
            max_new_tokens=50, 
            num_beams=5, 
            early_stopping=True
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_ozet = "A thief who infiltrates people’s minds to steal corporate secrets while they sleep."
print(f" Modelin Tahmini: {film_tahmin_et(test_ozet)}")

test_ozet_2 = "Spaceship ventures toward a black hole in order to rescue the planet."
print(f" Modelin Tahmini: {film_tahmin_et(test_ozet_2)}")

 Modelin Tahmini: Dream Sharing
 Modelin Tahmini: A Spacecraft travels to a black hole


In [26]:
def coklu_film_onerisi(izlenen_filmler_ozeti):
    model.eval()
    input_text = "recommend movie: " + izlenen_filmler_ozeti
    
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"], 
            max_new_tokens=50, 
            do_sample=True,      
            top_k=50,           
            top_p=0.95,          
            temperature=0.8,    
            num_return_sequences=1 
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

gecmis_ozetler = "A billionaire industrialist and master inventor, Tony Stark | Bruce Wayne becomes a masked vigilante who uses intelligence and technology to fight crime and protect city."

print(" Kullanıcının İzledikleri: Iron Man ve Batman")
print(f" Modelin Yeni Önerisi: {coklu_film_onerisi(gecmis_ozetler)}")

 Kullanıcının İzledikleri: Iron Man ve Batman
 Modelin Yeni Önerisi: The Dark Knight of Gotham City


In [ ]:
def coklu_film_onerisi(izlenen_filmler_ozeti):
    model.eval()
    
    input_text = "recommend movie: " + izlenen_filmler_ozeti
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            num_return_sequences=1
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


gecmis_ozetler = (
    "A computer hacker discovers that reality is a simulation and joins a rebellion against machines | "
    "A young blade runner uncovers a secret that could plunge society into chaos."
)

print(f"Modelin Yeni Önerisi: {coklu_film_onerisi(gecmis_ozetler)}")

